# 📖 Notebook 3: Market Data Streaming

Robinhood shows **live stock prices** that update in real time. This notebook explores
how an exchange trade feed flows through Kafka, gets processed, and is pushed to users
via Redis pub/sub.

## Learning Objectives

By the end of this notebook you will understand:
- How an exchange **trade feed** delivers price updates
- How **Kafka** acts as a durable buffer for high-throughput trade events
- How **Redis pub/sub** fans out price updates to connected servers
- Why **Server-Sent Events (SSE)** are preferred over polling or WebSockets for this use case
- How the full pipeline fits together: Exchange → Kafka → Processor → Redis → Client

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/robinhood
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `robinhood_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import time
import random
import threading
from datetime import datetime, timezone
from kafka import KafkaProducer, KafkaConsumer
from kafka.errors import NoBrokersAvailable

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "robinhood_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

KAFKA_BROKER = "localhost:9094"

# One topic per notebook run. Kafka topics are durable: pointed at a shared
# "trades" topic, a second run of this notebook would replay the *previous*
# run's trades and every count below would be wrong. A per-run topic makes
# "everything in the log" and "everything this run produced" the same set.
TRADES_TOPIC = f"trades-{int(time.time())}"


def ensure_topic(name: str, partitions: int = 1):
    """
    Create a topic up front instead of leaning on Kafka's auto-creation.

    Auto-creation happens *during* the first produce, so the first few records
    race the new partition's leader election — and `KafkaProducer` defaults to
    `retries=0`, so those records are dropped with no exception raised anywhere.
    That is how a feed "produces 15 trades" and the log ends up holding 11.
    """
    from kafka.admin import KafkaAdminClient, NewTopic
    from kafka.errors import TopicAlreadyExistsError

    admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
    try:
        admin.create_topics([NewTopic(name=name, num_partitions=partitions,
                                      replication_factor=1)])
    except TopicAlreadyExistsError:
        pass
    finally:
        admin.close()

def get_db():
    conn = psycopg2.connect(**DB_CONFIG)
    conn.autocommit = True
    return conn

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test all three connections
try:
    conn = get_db()
    conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis()
    r.ping()
    print("✅ Redis connected")
except Exception as e:
    print(f"❌ Redis failed: {e}")

try:
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )
    print(f"✅ Kafka connected — this run uses topic {TRADES_TOPIC!r}")
    producer.close()
except NoBrokersAvailable:
    print("❌ Kafka failed — run: docker compose up -d")

## 📚 The Architecture

Here's how live prices flow from the exchange to the user's screen:

```
  Exchange          Our Backend                              User
  ────────         ────────────                             ────
                                                              
  Trade Feed ─────► Kafka ─────► Price       Redis ──────► Symbol
  (external)        topic:       Processor    Pub/Sub       Service ───► SSE ──► App
                    trades       (consumer)   (fan-out)     (server)
                                    │
                                    └──► Postgres
                                         (price_history)
```

### Why This Pipeline?

| Component | Role | Why Not Skip It? |
|-----------|------|------------------|
| **Kafka** | Durable buffer for trade events | Absorbs bursts; replays on failure |
| **Price Processor** | Consumes trades, updates DB + cache | Single writer avoids conflicts |
| **Redis Pub/Sub** | Broadcasts to all symbol servers | Decouples processors from servers |
| **SSE** | Pushes to user's browser/app | No polling; server-initiated updates |

### Why SSE Instead of WebSockets?

- Price updates are **one-directional** (server → client)
- SSE works over standard HTTP (simpler load balancer config)
- Automatic reconnection built into the protocol
- WebSockets would be overkill — the client doesn't send price data back

## 1️⃣ Simulating the Exchange Trade Feed with Kafka

In the real world, the exchange pushes trades to us via a feed.  
We'll simulate this by producing trade events into a Kafka topic.

In [ ]:
# Load our symbols and their current prices
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT ticker, last_price_cents FROM symbols ORDER BY ticker")
symbols = {row['ticker']: row['last_price_cents'] for row in cur.fetchall()}
conn.close()

print("📊 Symbols loaded:")
for ticker, price in symbols.items():
    print(f"  {ticker:<6} ${price/100:.2f}")

In [ ]:
# Seeded so the "random" walk prints the same story on every run -- an unseeded
# feed makes every number in this notebook change from one execution to the next.
random.seed(20260821)


def simulate_exchange_feed(num_trades: int = 20, delay: float = 0.3, topic: str = None):
    """
    Simulate an exchange trade feed by producing random trades into Kafka.
    Each trade slightly moves the price up or down.
    """
    topic = topic or TRADES_TOPIC
    ensure_topic(topic)
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BROKER,
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        # A market-data feed cannot afford fire-and-forget produces:
        acks='all',   # the broker must have the record durably before it counts as sent
        retries=5,    # kafka-python defaults to 0 — one transient error loses the trade
        max_in_flight_requests_per_connection=1,  # retries must not reorder the tape
    )
    futures = []

    print(f"📡 Producing {num_trades} trades to Kafka topic {topic!r}...")
    print()

    for i in range(num_trades):
        ticker = random.choice(list(symbols.keys()))
        current_price = symbols[ticker]

        # Random price movement: ±0.5%. Round rather than truncate — int() always
        # rounds toward zero, which would drift every price steadily downward.
        change_pct = random.uniform(-0.005, 0.005)
        new_price = max(1, round(current_price * (1 + change_pct)))
        delta_cents = new_price - current_price
        symbols[ticker] = new_price  # update our local tracker

        trade = {
            "ticker": ticker,
            "price_cents": new_price,
            "quantity": random.randint(1, 100),
            "timestamp": datetime.now(timezone.utc).isoformat()
        }

        futures.append(producer.send(topic, value=trade))

        # Report the move that actually happened in cents, not the pre-rounding
        # percentage — a +0.001% tick can round to no move at all.
        direction = "📈" if delta_cents > 0 else ("📉" if delta_cents < 0 else "➡️")
        print(f"  {direction} {ticker:<6} ${new_price/100:.2f}  "
              f"({delta_cents:+d}¢)  ×{trade['quantity']} shares")

        time.sleep(delay)

    producer.flush()

    # Checking the futures is the whole point. `producer.send()` returns a
    # promise; a feed that never looks at it has no idea whether the exchange
    # data actually reached the log — it just prints a happy line either way.
    delivered = [f.get(timeout=15) for f in futures]
    producer.close()

    assert len(delivered) == num_trades, (
        f"sent {num_trades} trades but only {len(delivered)} were acknowledged"
    )
    print(f"\n✅ {num_trades} trades acknowledged by Kafka on topic {topic!r}")


simulate_exchange_feed(num_trades=15, delay=0.2)


## 2️⃣ Price Processor: Consuming from Kafka

The **price processor** reads trades from Kafka and does two things:
1. Updates the `last_price_cents` in Postgres (and stores price history)
2. Publishes the price update to Redis pub/sub so connected servers get notified

In [ ]:
def run_price_processor(max_messages: int = 15, timeout_ms: int = 15000):
    """
    Consume trades from Kafka, update Postgres, publish to Redis pub/sub.
    
    In production this runs continuously as a service.
    Here we process a fixed number for demonstration.
    """
    consumer = KafkaConsumer(
        TRADES_TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='earliest',
        consumer_timeout_ms=timeout_ms,
        group_id=f'price-processor-{int(time.time())}'
    )

    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    processed = 0
    print("⚙️  Price Processor running...")
    print()

    for message in consumer:
        trade = message.value
        ticker = trade['ticker']
        price = trade['price_cents']

        # 1. Update latest price in Postgres
        cur.execute("""
            UPDATE symbols SET last_price_cents = %s, updated_at = NOW()
            WHERE ticker = %s
            RETURNING id
        """, (price, ticker))
        result = cur.fetchone()

        if result:
            symbol_id = result[0]
            # Store in price history
            cur.execute("""
                INSERT INTO price_history (symbol_id, price_cents) VALUES (%s, %s)
            """, (symbol_id, price))

        # 2. Also cache latest price in Redis for fast lookups
        r.set(f"price:{ticker}", price)

        # 3. Publish to Redis pub/sub channel
        update_msg = json.dumps({
            "ticker": ticker,
            "price_cents": price,
            "timestamp": trade['timestamp']
        })
        r.publish(f"price:{ticker}", update_msg)

        processed += 1
        print(f"  ✅ [{processed}] {ticker} → ${price/100:.2f}  "
              f"(DB ✓, Cache ✓, Pub/Sub ✓)")

        if processed >= max_messages:
            break

    consumer.close()
    conn.close()
    print(f"\n⚙️  Processed {processed} trade(s)")
    return processed


processed = run_price_processor()

# The topic is fresh this run and the cell above produced exactly 15 trades, so
# reading from 'earliest' must yield exactly 15. Anything less means the
# consumer timed out mid-stream and the rest of the notebook is working off a
# partial picture.
assert processed == 15, f"expected to process all 15 produced trades, got {processed}"


## 3️⃣ Redis Pub/Sub: Fan-Out to Symbol Servers

In production, many **symbol service** servers maintain SSE connections to users.
Each server subscribes to Redis pub/sub channels for the symbols its connected users
care about.

```
  Price Processor ──publish──► Redis Pub/Sub
                                    │
                    ┌───────────────┼───────────────┐
                    ▼               ▼               ▼
              Symbol Server 1  Server 2        Server 3
              (AAPL, META)    (AAPL, TSLA)    (NVDA, META)
                    │               │               │
                    ▼               ▼               ▼
                Users A,B       Users C,D       Users E,F
```

Redis pub/sub is a good fit here because:
- It's **fire-and-forget** — no message persistence needed (prices are ephemeral)
- It's **fast** — sub-millisecond delivery on an idle server
- Servers subscribe only to channels they need

And the honest flip side, which matters as soon as the tape gets busy:
- There is **no backpressure and no redelivery**. A subscriber that reads too slowly
  fills its Redis output buffer, gets disconnected, and silently loses whatever it
  had not read. See **Backpressure** in the scaling notes at the end of this notebook.


In [ ]:
# Simulate a symbol server subscribing to price updates

received_updates = []  # collect updates for display
stop_flag = threading.Event()
subscribed_flag = threading.Event()

def symbol_server(subscribed_tickers: list):
    """
    Simulates a symbol service server that listens to Redis pub/sub
    for price updates and would push them to connected users via SSE.
    """
    r = get_redis()
    pubsub = r.pubsub()

    # Subscribe to channels for each ticker
    channels = [f"price:{t}" for t in subscribed_tickers]
    pubsub.subscribe(*channels)

    confirmed = 0
    for message in pubsub.listen():
        # Redis acknowledges each SUBSCRIBE with its own message. Anything
        # published before those land is dropped on the floor with no error
        # anywhere, so we signal "listening" only once they have all arrived —
        # a `time.sleep()` here is a guess, not a handshake.
        if message['type'] == 'subscribe':
            confirmed += 1
            if confirmed == len(channels):
                print(f"🖥️  Symbol server subscribed to: {', '.join(subscribed_tickers)}")
                subscribed_flag.set()
            continue
        if stop_flag.is_set():
            break
        if message['type'] == 'message':
            data = json.loads(message['data'])
            received_updates.append(data)

    pubsub.unsubscribe()
    pubsub.close()


# Start the "server" in a background thread
stop_flag.clear()
subscribed_flag.clear()
received_updates.clear()
server_thread = threading.Thread(
    target=symbol_server,
    args=(["AAPL", "TSLA", "NVDA"],)
)
server_thread.daemon = True
server_thread.start()
assert subscribed_flag.wait(timeout=15), "the symbol server never confirmed its subscriptions"

print("Server is listening... now let's publish some price updates.")


In [ ]:
# Publish some price updates (simulating what the price processor does)

r = get_redis()

test_updates = [
    {"ticker": "AAPL",  "price_cents": 19200, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "TSLA",  "price_cents": 24800, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "META",  "price_cents": 52500, "timestamp": datetime.now(timezone.utc).isoformat()},  # not subscribed!
    {"ticker": "NVDA",  "price_cents": 89000, "timestamp": datetime.now(timezone.utc).isoformat()},
    {"ticker": "AAPL",  "price_cents": 19250, "timestamp": datetime.now(timezone.utc).isoformat()},
]

for update in test_updates:
    r.publish(f"price:{update['ticker']}", json.dumps(update))
    print(f"  📡 Published: {update['ticker']} → ${update['price_cents']/100:.2f}")
    time.sleep(0.2)

time.sleep(0.5)  # let messages arrive

# Stop the server
stop_flag.set()
# Unblock the listener by publishing a dummy message
r.publish("price:AAPL", json.dumps({"ticker": "AAPL", "price_cents": 0, "timestamp": ""}))
server_thread.join(timeout=2)

# Show what the server received
# Filter out the dummy unblock message
real_updates = [u for u in received_updates if u['price_cents'] > 0]
print(f"\n🖥️  Symbol server received {len(real_updates)} updates:")
for u in real_updates:
    print(f"  {u['ticker']:<6} ${u['price_cents']/100:.2f}")

# The fan-out claim below is only interesting if it is actually true, so assert it:
# 4 of the 5 published updates were on subscribed channels, in publish order.
assert [u['ticker'] for u in real_updates] == ["AAPL", "TSLA", "NVDA", "AAPL"], real_updates
assert not any(u['ticker'] == 'META' for u in real_updates), \
    "META arrived on a channel this server never subscribed to"

print()
print("💡 Notice: META update was NOT received — the server wasn't subscribed to it!")
print("   This is the power of pub/sub: servers only get what they need.")


## 4️⃣ Full Pipeline Demo

Let's run the entire pipeline end-to-end:
1. Exchange produces trades → Kafka
2. Price processor reads Kafka → updates DB + publishes to Redis
3. Symbol server receives Redis pub/sub updates

We'll run each piece in a thread to simulate a real system.

> **Why this demo reads from `earliest` on a topic of its own.**
> The obvious way to say "start from now" is `auto_offset_reset='latest'`, and it
> is a race you cannot win from the outside. `'latest'` is not resolved when you
> build the consumer, and not even when the group join completes — it is resolved
> by a **ListOffsets round-trip issued on the first fetch after assignment**. If
> the producer sends anything between your readiness handshake and that
> round-trip, the end offset comes back *past* those records and the consumer
> never sees them. `seek_to_end()` does not help: it only re-arms the same lazy
> reset. The symptom is quietly losing the first message or two — "9 of the 10
> arrived" — with no error logged anywhere.
>
> Lossless requires a start point *you* control: an offset you committed, an
> explicit `seek()` to an offset you looked up yourself, or — simplest for a demo
> — a **fresh topic read from `earliest`**, where "everything in the log" and
> "everything this run produced" are the same set by construction. Real
> processors take the first route: they commit offsets and resume from them, so a
> restart neither loses nor replays.
>
> The Redis half has the mirror-image trap: a `PUBLISH` that arrives before a
> subscriber's `SUBSCRIBE` is registered is dropped silently. Both threads below
> therefore signal readiness on a real acknowledgement, never on a `sleep()`.


In [ ]:
# Full end-to-end pipeline

# A topic used by nothing else, so "read from the beginning" means exactly
# "read the trades this demo produced". See the note above.
PIPELINE_TOPIC = f"{TRADES_TOPIC}-pipeline"
TRADES_IN_DEMO = 10

# Create it before the consumer starts, so there is a real partition to be
# assigned to the moment it joins (and no auto-creation race for the producer).
ensure_topic(PIPELINE_TOPIC)
print(f"📼 Topic {PIPELINE_TOPIC!r} ready")

pipeline_updates = []
pipeline_stop = threading.Event()
processor_ready = threading.Event()
subscriber_ready = threading.Event()


def pipeline_subscriber(tickers):
    """Symbol server collecting updates."""
    r = get_redis()
    ps = r.pubsub()
    channels = [f"price:{t}" for t in tickers]
    ps.subscribe(*channels)

    confirmed = 0
    for msg in ps.listen():
        # Only report ready once Redis has acknowledged every SUBSCRIBE —
        # a publish that beats the subscription is lost with no error.
        if msg['type'] == 'subscribe':
            confirmed += 1
            if confirmed == len(channels):
                subscriber_ready.set()
            continue
        if pipeline_stop.is_set():
            break
        if msg['type'] == 'message':
            data = json.loads(msg['data'])
            if data.get('price_cents', 0) > 0:
                pipeline_updates.append({
                    **data,
                    "received_at": time.time()
                })
    ps.close()


def pipeline_processor():
    """Price processor consuming from Kafka."""
    consumer = KafkaConsumer(
        PIPELINE_TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        value_deserializer=lambda m: json.loads(m.decode('utf-8')),
        auto_offset_reset='earliest',
        consumer_timeout_ms=8000,
        group_id=f'pipeline-demo-{int(time.time())}'
    )
    # Wait for the group join so the three stages genuinely run concurrently.
    # Correctness does NOT depend on this: with 'earliest' on a topic nothing
    # else writes to, joining late still reads from offset 0 and sees every
    # trade. That is the whole point of not using 'latest' here.
    deadline = time.time() + 30
    while not consumer.assignment() and time.time() < deadline:
        consumer.poll(timeout_ms=200)
    processor_ready.set()

    conn = get_db()
    cur = conn.cursor()
    r = get_redis()

    for msg in consumer:
        trade = msg.value
        # Update DB
        cur.execute("UPDATE symbols SET last_price_cents = %s WHERE ticker = %s RETURNING id",
                    (trade['price_cents'], trade['ticker']))
        result = cur.fetchone()
        if result:
            cur.execute("INSERT INTO price_history (symbol_id, price_cents) VALUES (%s, %s)",
                        (result[0], trade['price_cents']))
        # Cache + pub/sub
        r.set(f"price:{trade['ticker']}", trade['price_cents'])
        r.publish(f"price:{trade['ticker']}", json.dumps(trade))

    consumer.close()
    conn.close()


# Start subscriber (all symbols) and wait for Redis to confirm the subscriptions
pipeline_stop.clear()
pipeline_updates.clear()
processor_ready.clear()
subscriber_ready.clear()
all_tickers = list(symbols.keys())
sub_thread = threading.Thread(target=pipeline_subscriber, args=(all_tickers,))
sub_thread.daemon = True
sub_thread.start()
assert subscriber_ready.wait(timeout=15), "the Redis subscriber never confirmed its subscriptions"

# Start processor and wait until it is genuinely attached to the topic
proc_thread = threading.Thread(target=pipeline_processor)
proc_thread.daemon = True
proc_thread.start()
assert processor_ready.wait(timeout=45), \
    "price processor never joined the Kafka consumer group — is the broker up?"

# Produce trades
print("🚀 Full Pipeline: Exchange → Kafka → Processor → Redis → Subscriber")
print("=" * 65)
simulate_exchange_feed(num_trades=TRADES_IN_DEMO, delay=0.3, topic=PIPELINE_TOPIC)

# Wait for the tail of the pipeline to drain
deadline = time.time() + 20
while len(pipeline_updates) < TRADES_IN_DEMO and time.time() < deadline:
    time.sleep(0.2)
pipeline_stop.set()

# Unblock subscriber
r_temp = get_redis()
r_temp.publish("price:AAPL", json.dumps({"ticker": "AAPL", "price_cents": 0, "timestamp": ""}))
sub_thread.join(timeout=5)
proc_thread.join(timeout=15)

print(f"\n📊 Subscriber received {len(pipeline_updates)} price updates")
print()

# Show unique tickers updated
ticker_counts = {}
for u in pipeline_updates:
    ticker_counts[u['ticker']] = ticker_counts.get(u['ticker'], 0) + 1

print("Updates per ticker:")
for ticker, count in sorted(ticker_counts.items()):
    print(f"  {ticker:<6} {count} update(s)")

# End-to-end conservation: every trade the exchange produced must come out the
# far end of the pipeline exactly once. Printing "received 0 updates" and moving
# on would look like a successful run while demonstrating nothing.
assert len(pipeline_updates) == TRADES_IN_DEMO, (
    f"produced {TRADES_IN_DEMO} trades but the subscriber saw {len(pipeline_updates)} — "
    "updates were dropped between Kafka, the processor and Redis pub/sub"
)
assert sum(ticker_counts.values()) == TRADES_IN_DEMO

# And Redis holds the latest price for every symbol that traded.
r_check = get_redis()
for ticker in ticker_counts:
    last_for_ticker = [u for u in pipeline_updates if u['ticker'] == ticker][-1]
    assert int(r_check.get(f"price:{ticker}")) == last_for_ticker['price_cents'], \
        f"Redis cache for {ticker} disagrees with the last update the subscriber saw"

print(f"\n✅ All {TRADES_IN_DEMO} trades made it Exchange → Kafka → Processor → Redis → Subscriber,")
print("   and the Redis cache matches the last price seen for every symbol.")


## 5️⃣ Fast Price Lookups from Redis Cache

While pub/sub delivers real-time updates, sometimes a user opens the app and needs
the *current* price immediately (before the next update arrives).

We store the latest price in Redis as a simple key-value for O(1) lookups.

In [ ]:
r = get_redis()

def get_live_prices(tickers: list) -> dict:
    """Fetch the latest prices from the Redis cache in ONE round-trip."""
    pipe = r.pipeline()
    for ticker in tickers:
        pipe.get(f"price:{ticker}")
    results = pipe.execute()

    prices = {}
    for ticker, val in zip(tickers, results):
        if val:
            prices[ticker] = int(val)
    return prices


# Measure latency
start = time.perf_counter()
prices = get_live_prices(all_tickers)
elapsed_ms = (time.perf_counter() - start) * 1000

assert prices, "Redis has no cached prices — run the price processor cells above first"

print(f"⚡ Fetched {len(prices)} prices in {elapsed_ms:.2f} ms")
print()
for ticker, price in sorted(prices.items()):
    print(f"  {ticker:<6} ${price/100:.2f}")

print()
print("💡 One pipelined round-trip for every symbol, instead of one round-trip each.")
print("   This is what powers the initial page load before SSE kicks in.")
print(f"   ({len(prices)} keys in {elapsed_ms:.2f} ms — the network hop dominates, not Redis.)")


## Bad Practice -> Best Practice: Polling vs Pub/Sub

A simple way to get prices to users would be to have the browser poll an HTTP endpoint every few seconds: `GET /price/AAPL` every 3 s. That's easy to build... and a disaster at scale.

- **Bad (polling)**: every client makes one request per interval whether or not the price changed. 1 million users polling every 3 s = **333 k req/s** of mostly-wasted traffic, and the user still sees updates up to 3 s late.
- **Good (pub/sub + SSE)**: the server pushes *only when the price actually changes*. One message per change, fanned out to just the subscribers who want that symbol. Sub-second latency, and near-zero traffic when the market is quiet.

Let's measure the two side by side.


In [ ]:
import statistics

r = get_redis()

# Load latest prices into Redis so the polling client has something to GET
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
cur.execute("SELECT ticker, last_price_cents FROM symbols")
for row in cur.fetchall():
    r.set(f"price:{row['ticker']}", row['last_price_cents'])
conn.close()

TICKER = "AAPL"
POLL_INTERVAL_S = 0.25     # the client polls 4x / second
DURATION_S = 2.0

# -------- BAD: polling --------
# One price change, and we measure how long the poller takes to notice it.
bad_requests = 0
start = time.perf_counter()
change_time = None
new_price = 99999
saw_at = None
last_seen = int(r.get(f"price:{TICKER}"))

while time.perf_counter() - start < DURATION_S:
    bad_requests += 1
    cur_price = int(r.get(f"price:{TICKER}"))
    if cur_price != last_seen and saw_at is None:
        saw_at = time.perf_counter()

    # Fire the change immediately AFTER a poll returns, so the poller has to
    # wait out a full interval before it can see it. That is not stacking the
    # deck: it is exactly the worst case a poll interval implies, and pinning
    # the change to a known point in the cycle is what makes this measurement
    # reproducible instead of a coin flip on where the change happens to land.
    if bad_requests == 4 and change_time is None:
        change_time = time.perf_counter()
        r.set(f"price:{TICKER}", new_price)

    time.sleep(POLL_INTERVAL_S)

assert saw_at is not None and change_time is not None, \
    "the poller never noticed the price change — the demo did not run"
bad_latency_ms = (saw_at - change_time) * 1000
print(f"BAD  polling:  {bad_requests} requests in {DURATION_S}s, latency to notice change = {bad_latency_ms:.0f} ms")

# Reset
r.set(f"price:{TICKER}", last_seen)

# -------- GOOD: pub/sub --------
good_messages = 0
pubsub_latency_ms = None
stop = threading.Event()

ready = threading.Event()

def subscriber():
    global good_messages, pubsub_latency_ms
    rr = get_redis()
    ps = rr.pubsub()
    ps.subscribe(f"price:{TICKER}")
    for msg in ps.listen():
        # Signal ready on Redis's SUBSCRIBE acknowledgement, not on a sleep:
        # a publish that beats the subscription is dropped silently.
        if msg['type'] == 'subscribe':
            ready.set()
            continue
        if stop.is_set():
            break
        if msg['type'] == 'message':
            good_messages += 1
            if pubsub_latency_ms is None:
                pubsub_latency_ms = (time.perf_counter() - publish_time) * 1000
    ps.close()

t = threading.Thread(target=subscriber, daemon=True)
t.start()
assert ready.wait(timeout=15), "the subscriber never confirmed its subscription"

# Publish a single price change -- the only "traffic" that happens
publish_time = time.perf_counter()
r.publish(f"price:{TICKER}", json.dumps({"ticker": TICKER, "price_cents": new_price}))

time.sleep(0.5)
stop.set()
# Unblock the listener
r.publish(f"price:{TICKER}", json.dumps({"ticker": TICKER, "price_cents": last_seen}))
t.join(timeout=2)

print(f"GOOD pub/sub:  {good_messages} message(s) delivered, latency to notice change = {pubsub_latency_ms:.1f} ms")
print()
print("Tip: polling sends traffic even when nothing happens, and its worst-case latency")
print("     is the poll interval. Pub/sub sends 1 message per real change and arrives in milliseconds.")

# Assert the lesson, so this section fails loudly if it ever stops reproducing it.
assert pubsub_latency_ms is not None, "the subscriber never received the published update"
assert good_messages == 1, \
    f"pub/sub should deliver exactly one message per real change, got {good_messages}"
assert bad_requests >= int(DURATION_S / POLL_INTERVAL_S) - 1, \
    f"expected ~{int(DURATION_S / POLL_INTERVAL_S)} polls in {DURATION_S}s, got {bad_requests}"
assert bad_requests > good_messages * 4, (
    f"polling made {bad_requests} requests to deliver the same one change that "
    f"pub/sub delivered in {good_messages} message"
)
# The change landed just after a poll, so the poller pays close to a full
# interval to see it — that IS the worst case polling signs up for.
assert POLL_INTERVAL_S * 1000 * 0.5 <= bad_latency_ms <= POLL_INTERVAL_S * 1000 * 2, (
    f"polling latency {bad_latency_ms:.0f} ms should be about one "
    f"{POLL_INTERVAL_S*1000:.0f} ms poll interval"
)
# Pub/sub is not bounded by any interval — it should be orders of magnitude faster.
assert pubsub_latency_ms < bad_latency_ms / 10, (
    f"pub/sub ({pubsub_latency_ms:.1f} ms) should notice the change far sooner "
    f"than polling ({bad_latency_ms:.0f} ms)"
)
print(f"\n✅ {bad_requests} polling requests vs {good_messages} pub/sub message, "
      f"{bad_latency_ms:.0f} ms vs {pubsub_latency_ms:.1f} ms to notice the change.")


## 📐 Scaling Considerations

In a real system with millions of users:

### 1. Kafka Partitioning
Partition the `trades` topic by symbol ticker. This ensures all trades for AAPL
go to the same partition → processed in order → no race conditions on price updates.

### 2. Redis Pub/Sub Channel Design
One channel per symbol (e.g., `price:AAPL`). Servers subscribe only to channels
that their connected users care about. If no user on a server watches AAPL,
that server doesn't subscribe to `price:AAPL`.

### 3. Throttling Updates to Clients
If AAPL trades 1000 times per second, we don't need to push every single trade.
The price processor can **batch** updates — e.g., publish at most once per 100ms
per symbol, sending only the latest price.

### 4. Sticky Sessions for SSE
SSE connections are long-lived. The load balancer must use **sticky sessions**
so a user's SSE connection always reaches the same server.

### 5. Backpressure — Pub/Sub Has None

This is the part the diagram hides. **Redis pub/sub never pushes back on the
publisher.** If a symbol server (or the SSE socket behind it) drains slower than
prices arrive, messages pile up in that subscriber's *client output buffer*, and
once it crosses `client-output-buffer-limit pubsub` (Redis ships with a 32 MB hard
limit and an 8 MB-for-60s soft limit) Redis **disconnects the subscriber**. The
publisher never learns anything went wrong; the subscriber just silently misses
updates until it reconnects. Nothing in this notebook demonstrates that, because
at 10 messages there is no queue to build — which is exactly why it needs saying.

For market data that is usually the *right* trade, because the correct response to
a slow consumer here is not "queue harder" — it is **conflation**: keep only the
latest price per symbol per client and drop the intermediate ticks. Nobody wants a
thirty-second-old quote delivered in order. In practice:

- throttle per symbol at the processor (item 3) so fan-out is bounded no matter how
  hot the tape gets;
- have each symbol server hold a `{symbol: latest_price}` map per SSE connection and
  flush it on a timer, instead of writing every message straight to the socket;
- watch `client_output_buffer` / `pubsub_clients` in Redis `INFO clients`, and treat
  subscriber disconnects as a **dropped-data alarm**, not a connection blip.

Kafka is the opposite and that is precisely why it sits *upstream*: it is a durable
log with real backpressure, so a slow processor falls behind in **offset** rather
than losing data, and can catch up or replay. Never put the durable half downstream
of the lossy half.


## 🧹 Cleanup

In [ ]:
# Clean up Redis price keys
r = get_redis()
keys = r.keys("price:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

# ...and the two per-run Kafka topics. They are cheap, but leaving one behind
# per execution is exactly the kind of litter that makes a later run confusing.
try:
    from kafka.admin import KafkaAdminClient

    admin = KafkaAdminClient(bootstrap_servers=KAFKA_BROKER)
    existing = admin.list_topics()
    doomed = [t for t in (TRADES_TOPIC, PIPELINE_TOPIC) if t in existing]
    if doomed:
        admin.delete_topics(doomed)
        print(f"🧹 Deleted Kafka topic(s): {', '.join(doomed)}")
    admin.close()
except Exception as e:
    print(f"ℹ️  Could not delete the run's Kafka topics ({e}) — harmless, they expire.")


## 📚 Summary

### Key Takeaways

1. **Kafka** buffers the exchange trade feed — durable, replayable, handles bursts.
2. **Price Processor** consumes from Kafka and writes to DB + Redis.
3. **Redis Pub/Sub** fans out price updates to only the servers that need them.
4. **Redis cache** stores latest prices for instant lookups on page load.
5. **SSE** pushes updates to clients — simpler than WebSockets for one-way data.
6. **Don't poll.** Pub/sub sends traffic only when prices actually change and delivers in milliseconds.
7. **Pub/sub has no backpressure.** A slow subscriber is disconnected and loses data
   silently, so put the durable log (Kafka) upstream and **conflate** — send the latest
   price per symbol, not every tick — instead of trying to queue your way out.

### For System Design Interviews

- Draw the full pipeline: Exchange → Kafka → Processor → Redis → SSE → Client
- Explain why SSE over WebSockets (unidirectional, simpler)
- Mention Kafka partitioning by symbol for ordering guarantees
- Discuss throttling/conflation to avoid overwhelming clients with rapid updates
- Note Redis pub/sub is fire-and-forget (no persistence, no backpressure) — fine for
  ephemeral price data, fatal for anything you have to deliver exactly once
- Sticky sessions needed for SSE at the load balancer

### What This Toy Does NOT Do

- **No SSE server.** We stop at the Redis subscriber; the HTTP layer, reconnection
  and `Last-Event-ID` replay are described but never built.
- **One Kafka partition, one processor.** Ordering per symbol is free here; with real
  partitioning you have to key by ticker to keep it.
- **No conflation or throttling in code** — the processor publishes every trade.
- **No market hours, halts, or stale-quote handling.**
- **Volumes are tiny.** 25 trades cannot surface buffer limits, consumer lag, or the
  fan-out cost of a million SSE connections.

### The Big Picture

Across all three notebooks, we've covered the core of a brokerage system:

| Notebook | What We Built |
|----------|---------------|
| 1. Order Matching Engine | Order types, order book, price-time priority, lifecycle, consistency |
| 2. Portfolio Tracking | Positions, P&L, one-transaction ledger writes, caching |
| 3. Market Data Streaming | Kafka pipeline, Redis pub/sub, live prices, backpressure limits |

Together these form the backbone of a system like Robinhood! 🎉
